In [1]:
# Imports and shared helpers

import pandas as pd
import numpy as np
import json
from collections import Counter
from insights_utils import (paths, compute_stylometric_features,
                            train_and_evaluate)
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Setup & feature assembly

In [2]:
# Load every input table and print shapes as a sanity check
data = pd.read_csv(paths["Dataset"])
specter = pd.read_csv(paths["SPECTER"])
entropy = pd.read_csv(paths["Internal_Entropy"])
graphsage = pd.read_csv(paths["GraphSage"])
targets = pd.read_csv(paths["target"])

print("Dataset shape:", data.shape)
print("Specter shape:", specter.shape)
print("Entropy shape:", entropy.shape)
print("GraphSage shape:", graphsage.shape)
print("Targets shape:", targets.shape)

Dataset shape: (131772, 9)
Specter shape: (131772, 770)
Entropy shape: (131772, 2)
GraphSage shape: (131772, 769)
Targets shape: (131772, 2)


In [3]:
# Confirm project_id is unique in every table before merging
print(data["project_id"].is_unique,
specter["project_id"].is_unique,
targets["project_id"].is_unique,
entropy["project_id"].is_unique,
graphsage["project_id"].is_unique)

True True True True True


In [4]:
# Inner-join everything on project_id into a single working df
df = data.merge(specter, on="project_id", how="inner")
df = df.merge(targets, on="project_id", how="inner")
df = df.merge(entropy, on="project_id", how="inner")
df = df.merge(graphsage, on="project_id", how="inner")
print("Merged shape:", df.shape)

Merged shape: (131772, 1548)


In [5]:
# Build combined title+abstract text field
df["combined_text"] = (
    df["title"].fillna("") + " " + df["abstract"].fillna("")
)

In [6]:
# Stylometric feature extractor (word/sentence stats, lexical diversity, complex-word ratio)
stylometric_df = df["combined_text"].apply(compute_stylometric_features)

In [7]:
# Attach stylometric columns onto df
stylometric_cols = [
    "word_count",
    "sentence_count",
    "avg_word_length",
    "avg_sentence_length",
    "lexical_diversity",
    "complex_word_ratio"
]

df = pd.concat([df, stylometric_df[stylometric_cols]], axis=1)

print("Updated df shape:", df.shape)

Updated df shape: (131772, 1555)


In [8]:
# Sanity check: no duplicate column names after all the merges
df.columns[df.columns.duplicated()]

Index([], dtype='object')

In [9]:
# Load fixed train/val/test split and slice df; assert sizes match
with open(paths["split"], "r") as f:
    split_dict = json.load(f)

train_ids = set(split_dict["train"])
val_ids   = set(split_dict["val"])
test_ids  = set(split_dict["test"])

train_df = df[df["project_id"].isin(train_ids)].copy()
val_df   = df[df["project_id"].isin(val_ids)].copy()
test_df  = df[df["project_id"].isin(test_ids)].copy()

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) == len(train_ids)
assert len(val_df) == len(val_ids)
assert len(test_df) == len(test_ids)

print("Split integrity passed.")

Train: (104685, 1555)
Val: (20129, 1555)
Test: (6958, 1555)
Split integrity passed.


In [10]:
# Collect the 768 SPECTER embedding column names
specter_cols = [c for c in df.columns if c.startswith("specter_")]

print("Specter feature count:", len(specter_cols))

Specter feature count: 768


In [11]:
# Entropy column (kept as a list so it composes with other feature lists)
ENTROPY_COL = ["entropy_score"]

In [12]:
# Feature columns for model 1: SPECTER + stylometric + entropy + target
TARGET_COL = "is_highly_cited"

MODEL_COLS = specter_cols + stylometric_cols + ENTROPY_COL + [TARGET_COL]

In [13]:
# Build X/y matrices for train, val, test
X_train = train_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_val = val_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL]

X_test = test_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print("X_train shape:", X_train.shape)

X_train shape: (104685, 775)


In [14]:
# Class balance check on training set
y_train.value_counts()

is_highly_cited
0    83360
1    21325
Name: count, dtype: int64

In [15]:
# Class weights to compensate for the ~4:1 imbalance
counter = Counter(y_train)
total = len(y_train)

class_weights = {
    0: total / (2 * counter[0]),
    1: total / (2 * counter[1])
}

print("Class weights:", class_weights)

Class weights: {0: 0.6279090690978887, 1: 2.4545134818288394}


## Train CatBoost models

In [16]:
# Model 1: SPECTER + stylometric + entropy
specter_entropy_features = specter_cols + stylometric_cols+ ENTROPY_COL

model_specter, results_specter = train_and_evaluate(
    X_train[specter_entropy_features], y_train,
    X_val[specter_entropy_features], y_val,
    X_test[specter_entropy_features], y_test,
    model_name="specter+entropy"
)


===== Training specter+entropy =====
0:	test: 0.5637838	best: 0.5637838 (0)	total: 377ms	remaining: 12m 34s
200:	test: 0.6925427	best: 0.6925427 (200)	total: 53.5s	remaining: 7m 59s
400:	test: 0.7029681	best: 0.7029857 (399)	total: 1m 41s	remaining: 6m 43s
600:	test: 0.7058213	best: 0.7063152 (554)	total: 2m 27s	remaining: 5m 43s
800:	test: 0.7079465	best: 0.7079836 (797)	total: 3m 15s	remaining: 4m 52s
1000:	test: 0.7085008	best: 0.7086057 (988)	total: 4m 1s	remaining: 4m
1200:	test: 0.7092951	best: 0.7094575 (1193)	total: 4m 45s	remaining: 3m 10s
1400:	test: 0.7093807	best: 0.7096907 (1305)	total: 5m 28s	remaining: 2m 20s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.709690696
bestIteration = 1305

Shrink model to first 1306 iterations.

Best threshold (val): 0.36206030150753765
Best validation F1: 0.43411849827156523

--- Validation ---
{'PR_AUC': 0.37092717505747913, 'F1': 0.43411849827156523, 'Recall': 0.6642066420664207, 'Precision': 0.32242655839503226}



In [17]:
# Collect the 768 GraphSAGE embedding column names
graphSage_cols = [c for c in df.columns if c.startswith("emb_")]

print("GraphSAGE feature count:", len(graphSage_cols))

GraphSAGE feature count: 768


In [18]:
# Feature columns for model 2: GraphSAGE + stylometric + target
TARGET_COL = "is_highly_cited"

MODEL_COLS = graphSage_cols + stylometric_cols  + [TARGET_COL]

In [19]:
# Rebuild X/y for the GraphSAGE feature set
X_train = train_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_val = val_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL]

X_test = test_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print("X_train shape:", X_train.shape)

X_train shape: (104685, 774)


In [20]:
# Model 2: GraphSAGE + stylometric
graphSage_features = graphSage_cols + stylometric_cols

model_graphSAGE, results_graphSAGE = train_and_evaluate(
    X_train[graphSage_features], y_train,
    X_val[graphSage_features], y_val,
    X_test[graphSage_features], y_test,
    model_name="graphSAGE"
)


===== Training graphSAGE =====
0:	test: 0.6156451	best: 0.6156451 (0)	total: 274ms	remaining: 9m 6s
200:	test: 0.6783829	best: 0.6783829 (200)	total: 46s	remaining: 6m 51s
400:	test: 0.6877179	best: 0.6877203 (396)	total: 1m 30s	remaining: 6m
600:	test: 0.6898048	best: 0.6898048 (600)	total: 2m 18s	remaining: 5m 23s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6902449703
bestIteration = 648

Shrink model to first 649 iterations.

Best threshold (val): 0.357537688442211
Best validation F1: 0.41736783616447054

--- Validation ---
{'PR_AUC': 0.3405191616909812, 'F1': 0.41736783616447054, 'Recall': 0.6467404674046741, 'Precision': 0.3080979725770538}

--- Test ---
{'PR_AUC': 0.3908484162210192, 'F1': 0.4560061208875287, 'Recall': 0.6250655479811221, 'Precision': 0.3589280337247817}


In [21]:
# Feature set for model 3: GraphSAGE + stylometric + entropy
TARGET_COL = "is_highly_cited"

MODEL_COLS = graphSage_cols + stylometric_cols  + [TARGET_COL]+ ENTROPY_COL

In [22]:
# Rebuild X/y for the GraphSAGE+entropy feature set
X_train = train_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_val = val_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_val = val_df[TARGET_COL]

X_test = test_df[MODEL_COLS].drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print("X_train shape:", X_train.shape)

X_train shape: (104685, 775)


In [23]:
# Model 3: GraphSAGE + stylometric + entropy
graphSage_entropy_features = graphSage_cols + stylometric_cols + ENTROPY_COL

model_graphSAGE, results_graphSAGE = train_and_evaluate(
    X_train[graphSage_entropy_features], y_train,
    X_val[graphSage_entropy_features], y_val,
    X_test[graphSage_entropy_features], y_test,
    model_name="graphSAGE+entropy"
)


===== Training graphSAGE+entropy =====
0:	test: 0.6241165	best: 0.6241165 (0)	total: 278ms	remaining: 9m 16s
200:	test: 0.6732239	best: 0.6732239 (200)	total: 48.2s	remaining: 7m 10s
400:	test: 0.6812215	best: 0.6812215 (400)	total: 1m 36s	remaining: 6m 24s
600:	test: 0.6850888	best: 0.6851423 (595)	total: 2m 21s	remaining: 5m 29s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.6856294992
bestIteration = 639

Shrink model to first 640 iterations.

Best threshold (val): 0.321356783919598
Best validation F1: 0.41168768520432775

--- Validation ---
{'PR_AUC': 0.3396212951556349, 'F1': 0.41168768520432775, 'Recall': 0.734809348093481, 'Precision': 0.28594677388474055}

--- Test ---
{'PR_AUC': 0.3825264898015111, 'F1': 0.46156439649359404, 'Recall': 0.7178814892501311, 'Precision': 0.34012422360248445}


## Embedding signal analysis

In [24]:
# Map project_id -> is_highly_cited label
label_dict = dict(zip(targets['project_id'],
                      targets['is_highly_cited']))

In [25]:
# Observed edge homophily: fraction of mutual-kNN edges where both endpoints share the label
same = 0
total = 0
mut_edges = np.load(r"kg/mutual_edges@20.npy")

for u, v in mut_edges:
    if u in label_dict and v in label_dict:
        total += 1
        if label_dict[u] == label_dict[v]:
            same += 1

observed_homophily = same / total
print(f"Observed Edge Homophily: {observed_homophily:.4f}")
print(f"Total usable edges: {total}")

Observed Edge Homophily: 0.6643
Total usable edges: 516583


In [26]:
# Expected homophily if labels were random — baseline to compare against the observed value
p = targets['is_highly_cited'].mean()
expected_random = p**2 + (1 - p)**2

print(f"Positive class fraction: {p:.4f}")
print(f"Expected random homophily: {expected_random:.4f}")

Positive class fraction: 0.2072
Expected random homophily: 0.6715


In [27]:
# Per-dimension Pearson correlation of GraphSAGE embeddings with the target (signal check)
# Merge to ensure embeddings align with the correct label by project_id
merged_gs_trgt = (
    graphsage.merge(
        targets[["project_id", "is_highly_cited"]],
        on="project_id",
        how="inner",
        validate="one_to_one",
    )
)

emb_cols = [c for c in merged_gs_trgt.columns if c.startswith("emb_")]

# Pearson correlation of each embedding dimension vs the binary target
corrs = merged_gs_trgt[emb_cols].corrwith(merged_gs_trgt["is_highly_cited"], method="pearson")

avg_corr = float(corrs.mean())
max_corr = float(corrs.max())
min_corr = float(corrs.min())

max_dim = str(corrs.idxmax())
min_dim = str(corrs.idxmin())

print(f"Computed correlations for {len(corrs)} embedding dimensions.")
print(f"Average corr: {avg_corr:.6f}")
print(f"Max corr    : {max_corr:.6f} ({max_dim})")
print(f"Min corr    : {min_corr:.6f} ({min_dim})")

Computed correlations for 768 embedding dimensions.
Average corr: 0.001058
Max corr    : 0.177439 (emb_405)
Min corr    : -0.171100 (emb_254)


In [28]:
# Preview SPECTER embeddings
specter.head()

,project_id,original_id_clean,specter_embedding_0,specter_embedding_1,specter_embedding_2,specter_embedding_3,specter_embedding_4,specter_embedding_5,specter_embedding_6,specter_embedding_7,...,specter_embedding_758,specter_embedding_759,specter_embedding_760,specter_embedding_761,specter_embedding_762,specter_embedding_763,specter_embedding_764,specter_embedding_765,specter_embedding_766,specter_embedding_767
0,0,O02-2002,-0.362291,1.048671,0.595254,-0.452969,0.130679,-0.753185,-0.559871,0.633012,...,0.073975,-1.110611,0.944754,0.682057,0.666358,-0.090816,-0.156717,0.244948,0.692705,0.590688
1,2,R13-1042,-0.439786,0.765530,0.333816,-0.228252,0.821623,-0.042048,0.395873,0.196598,...,0.007341,-1.544107,1.061482,0.893701,0.390635,0.195787,0.161993,-0.035524,1.541261,1.091505
2,3,W05-0819,-0.009004,0.396810,0.429890,-0.111672,0.422623,-0.328100,-0.175239,1.196574,...,-0.286731,-1.943584,0.421188,1.321711,-0.241398,-0.169670,0.614148,-0.250611,1.043618,1.471813
3,5,R13-1044,0.019991,0.770271,-0.205909,-0.361313,-0.160810,-0.339320,0.291656,0.741180,...,0.255495,-1.617394,0.652282,0.675482,-0.270218,-0.458816,0.000720,-0.049819,1.302014,-0.347194
4,6,W05-0818,0.165885,0.157862,0.430685,0.589608,0.827854,-0.669882,0.357143,0.577054,...,-0.308747,-1.909491,0.734672,0.635045,-0.447741,-0.159738,0.051587,0.188710,0.718381,1.436688


In [29]:
# Per-dimension Pearson correlation of SPECTER embeddings with the target (signal check)
# Merge to ensure embeddings align with the correct label by project_id
merged_spe_trgt = (
    specter.merge(
        targets[["project_id", "is_highly_cited"]],
        on="project_id",
        how="inner",
        validate="one_to_one",
    )
)

emb_cols = [c for c in merged_spe_trgt.columns if c.startswith("specter_embedding_")]

# Pearson correlation of each embedding dimension vs the binary target
corrs = merged_spe_trgt[emb_cols].corrwith(merged_spe_trgt["is_highly_cited"], method="pearson")

avg_corr = float(corrs.mean())
max_corr = float(corrs.max())
min_corr = float(corrs.min())

max_dim = str(corrs.idxmax())
min_dim = str(corrs.idxmin())

print(f"Computed correlations for {len(corrs)} embedding dimensions.")
print(f"Average corr: {avg_corr:.6f}")
print(f"Max corr    : {max_corr:.6f} ({max_dim})")
print(f"Min corr    : {min_corr:.6f} ({min_dim})")

Computed correlations for 768 embedding dimensions.
Average corr: 0.000378
Max corr    : 0.121707 (specter_embedding_591)
Min corr    : -0.119970 (specter_embedding_551)
